## Imports and settings

In [1]:
from src.pipelines import *

## Define pipeline config

In [2]:
PIPELINE_CONFIG = {
    'pipeline_drop': {
        'steps': [
            {'registry': 'custom', 'key': 'drop', 'param': {}},
        ]
    },
    'pipeline_onehot': {
        'steps': [
            {'registry': 'imputer', 'key': 'simple', 'param': {'strategy': 'constant', 'fill_value': LABEL_UNKNOWN}},
            {'registry': 'encoder', 'key': 'onehot', 'param': {'handle_unknown': 'ignore', 'sparse_output': False}},
        ]
    },
    'pipeline_admission_type': {
        'steps': [
            {'registry': 'imputer', 'key': 'simple', 'param': {'strategy': 'constant', 'fill_value': LABEL_UNKNOWN}},
            {'registry': 'custom', 'key': 'admission_type_group', 'param': {}},
            {'registry': 'encoder', 'key': 'onehot', 'param': {'handle_unknown': 'ignore', 'sparse_output': False}},
        ]
    },
    'pipeline_discharge': {
        'steps': [
            {'registry': 'imputer', 'key': 'simple', 'param': {'strategy': 'constant', 'fill_value': LABEL_UNKNOWN}},
            {'registry': 'custom', 'key': 'discharge_group', 'param': {}},
            {'registry': 'encoder', 'key': 'onehot', 'param': {'handle_unknown': 'ignore', 'sparse_output': False}},
        ]
    },
    'pipeline_admission_src': {
        'steps': [
            {'registry': 'imputer', 'key': 'simple', 'param': {'strategy': 'constant', 'fill_value': LABEL_UNKNOWN}},
            {'registry': 'custom', 'key': 'admission_src_group', 'param': {}},
            {'registry': 'encoder', 'key': 'onehot', 'param': {'handle_unknown': 'ignore', 'sparse_output': False}},
        ]
    },
    'pipeline_medical': {
        'steps': [
            {'registry': 'imputer', 'key': 'simple', 'param': {'strategy': 'constant', 'fill_value': LABEL_UNKNOWN}},
            {'registry': 'custom', 'key': 'medical_group', 'param': {}},
            {'registry': 'encoder', 'key': 'onehot', 'param': {'handle_unknown': 'ignore', 'sparse_output': False}},
        ]
    },
    'pipeline_icd9': {
        'steps': [
            {'registry': 'imputer', 'key': 'simple', 'param': {'strategy': 'constant', 'fill_value': LABEL_UNKNOWN}},
            {'registry': 'custom', 'key': 'icd9_group', 'param': {}},
            {'registry': 'encoder', 'key': 'onehot', 'param': {'handle_unknown': 'ignore', 'sparse_output': False}},
        ]
    },
    'pipeline_prescript': {
        'steps': [
            {'registry': 'custom', 'key': 'high_no_drop', 'param': {'th': 1.}},
            {'registry': 'custom', 'key': 'prescription_group', 'param': {}},
        ]
    },
    'pipeline_standard': {
        'steps': [
            {'registry': 'scaler', 'key': 'standard', 'param': {}},
        ]
    },
}

## Define column config (assign pipeline to each column)

In [3]:
COLUMN_CONFIG_BASE = {
    # EDA: missing data analysis
    'encounter_id'            : 'pipeline_drop',
    # 'patient_nbr'             : 'pipeline_drop',          # Already dropped in split data
    'weight'                  : 'pipeline_drop',
    'payer_code'              : 'pipeline_drop',            # or 'impute_unknown'; grouping + onehot_encoding,
    'change'                  : 'pipeline_drop',
    'diabetesMed'             : 'pipeline_drop',

    # EDA: cardinality analysis
    'diag_1'                  : 'pipeline_icd9',            # or (grouping +) embedding (for NN)],
    'diag_2'                  : 'pipeline_icd9',            # or (grouping +) embedding (for NN)],
    'diag_3'                  : 'pipeline_icd9',            # or (grouping +) embedding (for NN)],
    'max_glu_serum'           : 'pipeline_onehot',          # or ordinal_encoding for NN
    'A1Cresult'               : 'pipeline_onehot',          # or ordinal_encoding for NN

    # EDA: age, gender and race analysis
    'race'                    : 'pipeline_onehot',
    'gender'                  : 'pipeline_onehot',
    'age'                     : 'pipeline_onehot',          # or ordinal_encoding for NN

    # EDA: medical specialty and operations
    'admission_type_id'       : 'pipeline_admission_type',  # or grouping + ordinal_encoding
    'discharge_disposition_id': 'pipeline_discharge',
    'admission_source_id'     : 'pipeline_admission_src',
    'medical_specialty'       : 'pipeline_medical',         # or (grouping +) embedding (for NN),

    # EDA: prescription data analysis
    'metformin'               : 'pipeline_prescript',
    'repaglinide'             : 'pipeline_prescript',
    'nateglinide'             : 'pipeline_prescript',
    'chlorpropamide'          : 'pipeline_prescript',
    'glimepiride'             : 'pipeline_prescript',
    'acetohexamide'           : 'pipeline_prescript',
    'glipizide'               : 'pipeline_prescript',
    'glyburide'               : 'pipeline_prescript',
    'tolbutamide'             : 'pipeline_prescript',
    'pioglitazone'            : 'pipeline_prescript',
    'rosiglitazone'           : 'pipeline_prescript',
    'acarbose'                : 'pipeline_prescript',
    'miglitol'                : 'pipeline_prescript',
    'troglitazone'            : 'pipeline_prescript',
    'tolazamide'              : 'pipeline_prescript',
    'examide'                 : 'pipeline_prescript',
    'citoglipton'             : 'pipeline_prescript',
    'insulin'                 : 'pipeline_prescript',
    'glyburide-metformin'     : 'pipeline_prescript',
    'glipizide-metformin'     : 'pipeline_prescript',
    'glimepiride-pioglitazone': 'pipeline_prescript',
    'metformin-rosiglitazone' : 'pipeline_prescript',
    'metformin-pioglitazone'  : 'pipeline_prescript',

    # EDA: numerical feature analysis
    'time_in_hospital'        : 'pipeline_standard',
    'num_lab_procedures'      : 'pipeline_standard',
    'num_procedures'          : 'pipeline_standard',
    'num_medications'         : 'pipeline_standard',
    'number_outpatient'       : 'pipeline_standard',
    'number_emergency'        : 'pipeline_standard',
    'number_inpatient'        : 'pipeline_standard',
    'number_diagnoses'        : 'pipeline_standard',
}

## Load shuffled data and stratified data

In [4]:
data_shuffle, data_stratify = load_data()

print('Keys (data_shuffle):', data_shuffle.keys())
print('Keys (data_stratify):', data_stratify.keys())

Keys (data_shuffle): dict_keys(['X_train_for_cv', 'X_train_mini', 'X_val', 'X_test', 'y_train_for_cv', 'y_train_mini', 'y_val', 'y_test'])
Keys (data_stratify): dict_keys(['X_train_for_cv', 'X_train_mini', 'X_val', 'X_test', 'y_train_for_cv', 'y_train_mini', 'y_val', 'y_test'])


## Build pipeline processors

In [5]:
pipeline_shuffle_base = PipelineBuilder(PIPELINE_CONFIG, COLUMN_CONFIG_BASE).build()
pipeline_stratify_base = PipelineBuilder(PIPELINE_CONFIG, COLUMN_CONFIG_BASE).build()

## Execute pipelines for input features (base)

In [6]:
data_shuffle_base  = dict()
data_stratify_base = dict()

# data_shuffle - base case
data_shuffle_base['X_train_mini']  = pipeline_shuffle_base.fit_transform(data_shuffle['X_train_mini'])
data_shuffle_base['X_val']         = pipeline_shuffle_base.transform(data_shuffle['X_val'])
data_shuffle_base['X_test']        = pipeline_shuffle_base.transform(data_shuffle['X_test'])

# data_stratify - base case
data_stratify_base['X_train_mini'] = pipeline_stratify_base.fit_transform(data_stratify['X_train_mini'])
data_stratify_base['X_val']        = pipeline_stratify_base.transform(data_stratify['X_val'])
data_stratify_base['X_test']       = pipeline_stratify_base.transform(data_stratify['X_test'])


## Save outputs to folder

In [8]:
SAVE_OUTPUT = True

PATH_DATA_PROCESSED_SHUFFLE = Path('../data/final_processed/base_groupshuffle')
PATH_DATA_PROCESSED_STRATIFY = Path('../data/final_processed/base_stratified')

if SAVE_OUTPUT:
    for k, v in data_shuffle_base.items():
        if k in ['X_train_mini', 'X_val', 'X_test']:
            v.to_csv(PATH_DATA_PROCESSED_SHUFFLE / f'{k + '.csv'}', index=False)
    for k, v in data_stratify_base.items():
        if k in ['X_train_mini', 'X_val', 'X_test']:
            v.to_csv(PATH_DATA_PROCESSED_STRATIFY / f'{k + '.csv'}', index=False)